In [ ]:
!pip install --upgrade open_clip_torch transformers ftfy -q

import os
import shutil
import gc
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import v2
import open_clip
from transformers import CLIPModel, CLIPTokenizer

val_path = "/kaggle/input/datasets/titericz/imagenet1k-val/imagenet-val"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TARGET_TEXT = "a clean graphic of the single large word 'DOG' centered"
EPSILON = 8 / 255
EPOCHS = 20
ALPHA = EPSILON * 2.5 / EPOCHS

class Catch_Image(Dataset):
    def __init__(self, root_path, transform, num_samples, skip_samples=0):
        self.root_path = root_path
        self.transform = transform
        self.num_samples = num_samples
        self.skip_samples = skip_samples
        raw_classes = sorted([d for d in os.listdir(root_path) if os.path.isdir(os.path.join(root_path, d))])
        valid_classes = set()
        img_paths = []
        label_texts = []
        for class_name in raw_classes:
            class_path = os.path.join(root_path, class_name)
            files_in_class = sorted([f for f in os.listdir(class_path) if os.path.isfile(os.path.join(class_path, f))])
            selected_files = files_in_class[self.skip_samples : self.skip_samples + self.num_samples]
            if len(selected_files) == 0:
                continue
            for fname in selected_files:
                img_paths.append(os.path.join(class_path, fname))
                label_texts.append(class_name)
            valid_classes.add(class_name)
        self.classes = sorted(list(valid_classes))
        class_to_idx = {class_name: idx for idx, class_name in enumerate(self.classes)}
        self.img_paths = img_paths
        self.labels = [class_to_idx[text_label] for text_label in label_texts]

    def __getitem__(self, index):
        img_path = self.img_paths[index]
        image = Image.open(img_path).convert("RGB")
        return self.transform(image), self.labels[index]

    def __len__(self):
        return len(self.img_paths)

raw_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor()
])

train_dataset = Catch_Image(root_path=val_path, transform=raw_transform, num_samples=3, skip_samples=0)
test_dataset = Catch_Image(root_path=val_path, transform=raw_transform, num_samples=1, skip_samples=3)

train_loader = DataLoader(dataset=train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(dataset=test_dataset, batch_size=16, shuffle=False)

class CLIPWrapper(nn.Module):
    def __init__(self, model_name="openai/clip-vit-base-patch32"):
        super().__init__()
        self.model = CLIPModel.from_pretrained(model_name).to(device).eval()
        self.register_buffer('mean', torch.tensor([0.48145466, 0.4578275, 0.40821073], device=device).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.26862954, 0.26130258, 0.27577711], device=device).view(1, 3, 1, 1))

class OpenCLIPWrapper(nn.Module):
    def __init__(self, arch, dataset, mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711)):
        super().__init__()
        self.model, _, _ = open_clip.create_model_and_transforms(arch, pretrained=dataset)
        self.model = self.model.to(device).eval()
        self.register_buffer('mean', torch.tensor(mean, device=device).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor(std, device=device).view(1, 3, 1, 1))

torch.cuda.empty_cache()
gc.collect()

vlm_models = {
    'clip': CLIPWrapper("openai/clip-vit-base-patch32").to(device),
    'open_clip': OpenCLIPWrapper('ViT-B-32', 'laion2b_s34b_b79k').to(device),
    'metaclip': OpenCLIPWrapper('ViT-B-32-quickgelu', 'metaclip_fullcc').to(device),
    'siglip': OpenCLIPWrapper('ViT-B-16-SigLIP', 'webli', mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)).to(device),
    'eva_clip': OpenCLIPWrapper('EVA02-B-16', 'merged2b_s8b_b131k').to(device)
}

for model in vlm_models.values():
    for p in model.parameters():
        p.requires_grad = False

def get_clip_image_embed(clip_wrapper, img_tensor):
    x_norm = (img_tensor - clip_wrapper.mean) / clip_wrapper.std
    model = clip_wrapper.model
    if hasattr(model, 'visual_projection'):
        vision_outputs = model.vision_model(pixel_values=x_norm)
        res = model.visual_projection(vision_outputs.pooler_output)
    else:
        raw = model.encode_image(x_norm)
        res = raw if isinstance(raw, torch.Tensor) else raw[0]
    return F.normalize(res, dim=-1)

def get_clip_text_embed(clip_wrapper, tokenizer_fn, text):
    model = clip_wrapper.model
    with torch.no_grad():
        if hasattr(model, 'text_projection') and hasattr(model, 'text_model'):
            tokens = tokenizer_fn([text], padding=True, return_tensors="pt").to(device)
            text_outputs = model.text_model(**tokens)
            res = model.text_projection(text_outputs.pooler_output)
        else:
            tokens = tokenizer_fn([text]).to(device)
            raw = model.encode_text(tokens)
            res = raw if isinstance(raw, torch.Tensor) else raw[0]
        return F.normalize(res, dim=-1)

def get_tokenizer_for_model(model_name):
    if model_name == 'clip':
        tok = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
        return lambda texts, **kwargs: tok(texts, **kwargs)
    else:
        arch_map = {'open_clip': 'ViT-B-32', 'siglip': 'ViT-B-16-SigLIP', 'eva_clip': 'EVA02-B-16', 'metaclip': 'ViT-B-32-quickgelu'}
        tok = open_clip.get_tokenizer(arch_map[model_name])
        return lambda texts: tok(texts)

tokenizers = {name: get_tokenizer_for_model(name) for name in vlm_models.keys()}
text_target_embeds = {name: get_clip_text_embed(model, tokenizers[name], TARGET_TEXT) for name, model in vlm_models.items()}

def get_gaussian_kernel(kernel_size=15, sigma=3.0, channels=3, device='cpu'):
    x = torch.arange(kernel_size, dtype=torch.float32, device=device)
    mean = (kernel_size - 1) / 2.0
    gaussian1d = torch.exp(-((x - mean) / sigma) ** 2 / 2.0)
    gaussian2d = gaussian1d.unsqueeze(1) * gaussian1d.unsqueeze(0)
    gaussian2d = gaussian2d / gaussian2d.sum()
    kernel = gaussian2d.view(1, 1, kernel_size, kernel_size).repeat(channels, 1, 1, 1)
    return kernel

def evaluate_val_loss(model, dataloader, target_embed, v):
    losses = []
    with torch.no_grad():
        for imgs, _ in dataloader:
            imgs = imgs.to(device)
            perturbed = torch.clamp(imgs + v, 0.0, 1.0)
            embed = get_clip_image_embed(model, perturbed)
            sim = F.cosine_similarity(embed, target_embed, dim=-1).mean()
            losses.append(-sim.item())
    return float(np.mean(losses))

def train_vanilla(model, target_embed, train_loader, val_loader, eps, alpha, epochs):
    v = torch.zeros((1, 3, 224, 224), device=device, requires_grad=True)
    best_v = v.detach().clone()
    best_val_loss = float('inf')
    history = []

    for epoch in range(epochs):
        running_loss = []
        pbar = tqdm(train_loader, desc=f"Vanilla Epoch {epoch+1}/{epochs}", leave=False)
        for imgs, _ in pbar:
            imgs = imgs.to(device)
            perturbed = torch.clamp(imgs + v, 0.0, 1.0)
            embed = get_clip_image_embed(model, perturbed)
            sim = F.cosine_similarity(embed, target_embed, dim=-1).mean()
            total_loss = -sim
            total_loss.backward()
            with torch.no_grad():
                v -= alpha * v.grad.sign()
                v.clamp_(-eps, eps)
                v.grad.zero_()
            running_loss.append(total_loss.item())
            pbar.set_postfix({'train_loss': f"{total_loss.item():.4f}"})

        train_loss = float(np.mean(running_loss))
        val_loss = evaluate_val_loss(model, val_loader, target_embed, v)
        history.append({'epoch': epoch + 1, 'train_loss': train_loss, 'val_loss': val_loss})
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_v = v.detach().clone()

    return best_v, history

def train_only_tifgsm(model, target_embed, train_loader, val_loader, eps, alpha, epochs, kernel_size=15, sigma=3.0):
    v = torch.zeros((1, 3, 224, 224), device=device, requires_grad=True)
    gaussian_kernel = get_gaussian_kernel(kernel_size=kernel_size, sigma=sigma, channels=3, device=device)
    best_v = v.detach().clone()
    best_val_loss = float('inf')
    history = []

    for epoch in range(epochs):
        running_loss = []
        pbar = tqdm(train_loader, desc=f"Only TI Epoch {epoch+1}/{epochs}", leave=False)
        for imgs, _ in pbar:
            imgs = imgs.to(device)
            perturbed = torch.clamp(imgs + v, 0.0, 1.0)
            embed = get_clip_image_embed(model, perturbed)
            sim = F.cosine_similarity(embed, target_embed, dim=-1).mean()
            total_loss = -sim
            total_loss.backward()
            with torch.no_grad():
                grad = v.grad
                grad = F.conv2d(grad, gaussian_kernel, padding=kernel_size//2, groups=3)
                grad_l1_norm = torch.sum(torch.abs(grad))
                grad = grad / (grad_l1_norm + 1e-12)
                v -= alpha * grad.sign()
                v.clamp_(-eps, eps)
                v.grad.zero_()
            running_loss.append(total_loss.item())
            pbar.set_postfix({'train_loss': f"{total_loss.item():.4f}"})

        train_loss = float(np.mean(running_loss))
        val_loss = evaluate_val_loss(model, val_loader, target_embed, v)
        history.append({'epoch': epoch + 1, 'train_loss': train_loss, 'val_loss': val_loss})
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_v = v.detach().clone()

    return best_v, history

def train_only_eot(model, target_embed, train_loader, val_loader, eps, alpha, epochs):
    v = torch.zeros((1, 3, 224, 224), device=device, requires_grad=True)
    eot_transform = v2.Compose([
        v2.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.8, 1.2)),
    ])
    best_v = v.detach().clone()
    best_val_loss = float('inf')
    history = []

    for epoch in range(epochs):
        running_loss = []
        pbar = tqdm(train_loader, desc=f"Only EOT Epoch {epoch+1}/{epochs}", leave=False)
        for imgs, _ in pbar:
            imgs = imgs.to(device)
            perturbed = torch.clamp(imgs + v, 0.0, 1.0)
            transformed = eot_transform(perturbed)
            embed = get_clip_image_embed(model, transformed)
            sim = F.cosine_similarity(embed, target_embed, dim=-1).mean()
            total_loss = -sim
            total_loss.backward()
            with torch.no_grad():
                v -= alpha * v.grad.sign()
                v.clamp_(-eps, eps)
                v.grad.zero_()
            running_loss.append(total_loss.item())
            pbar.set_postfix({'train_loss': f"{total_loss.item():.4f}"})

        train_loss = float(np.mean(running_loss))
        val_loss = evaluate_val_loss(model, val_loader, target_embed, v)
        history.append({'epoch': epoch + 1, 'train_loss': train_loss, 'val_loss': val_loss})
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_v = v.detach().clone()

    return best_v, history

def train_both_ti_eot(model, target_embed, train_loader, val_loader, eps, alpha, epochs, kernel_size=15, sigma=3.0):
    v = torch.zeros((1, 3, 224, 224), device=device, requires_grad=True)
    gaussian_kernel = get_gaussian_kernel(kernel_size=kernel_size, sigma=sigma, channels=3, device=device)
    eot_transform = v2.Compose([
        v2.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.8, 1.2)),
    ])
    best_v = v.detach().clone()
    best_val_loss = float('inf')
    history = []

    for epoch in range(epochs):
        running_loss = []
        pbar = tqdm(train_loader, desc=f"Both TI+EOT Epoch {epoch+1}/{epochs}", leave=False)
        for imgs, _ in pbar:
            imgs = imgs.to(device)
            perturbed = torch.clamp(imgs + v, 0.0, 1.0)
            transformed = eot_transform(perturbed)
            embed = get_clip_image_embed(model, transformed)
            sim = F.cosine_similarity(embed, target_embed, dim=-1).mean()
            total_loss = -sim
            total_loss.backward()
            with torch.no_grad():
                grad = v.grad
                grad = F.conv2d(grad, gaussian_kernel, padding=kernel_size//2, groups=3)
                grad_l1_norm = torch.sum(torch.abs(grad))
                grad = grad / (grad_l1_norm + 1e-12)
                v -= alpha * grad.sign()
                v.clamp_(-eps, eps)
                v.grad.zero_()
            running_loss.append(total_loss.item())
            pbar.set_postfix({'train_loss': f"{total_loss.item():.4f}"})

        train_loss = float(np.mean(running_loss))
        val_loss = evaluate_val_loss(model, val_loader, target_embed, v)
        history.append({'epoch': epoch + 1, 'train_loss': train_loss, 'val_loss': val_loss})
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_v = v.detach().clone()

    return best_v, history

save_dir = "/kaggle/working/ablation_uaps"
os.makedirs(save_dir, exist_ok=True)
all_histories = {}

for model_name, model in vlm_models.items():
    target_embed = text_target_embeds[model_name]

    uap_vanilla, hist_vanilla = train_vanilla(model, target_embed, train_loader, val_loader, eps=EPSILON, alpha=ALPHA, epochs=EPOCHS)
    uap_ti, hist_ti = train_only_tifgsm(model, target_embed, train_loader, val_loader, eps=EPSILON, alpha=ALPHA, epochs=EPOCHS)
    uap_eot, hist_eot = train_only_eot(model, target_embed, train_loader, val_loader, eps=EPSILON, alpha=ALPHA, epochs=EPOCHS)
    uap_both, hist_both = train_both_ti_eot(model, target_embed, train_loader, val_loader, eps=EPSILON, alpha=ALPHA, epochs=EPOCHS)

    uap_dict = {
        f"{model_name}_uap_vanilla": (uap_vanilla, hist_vanilla),
        f"{model_name}_uap_only_ti": (uap_ti, hist_ti),
        f"{model_name}_uap_only_eot": (uap_eot, hist_eot),
        f"{model_name}_uap_both_ti_eot": (uap_both, hist_both)
    }

    for name, (tensor, hist) in uap_dict.items():
        torch.save(tensor.cpu(), os.path.join(save_dir, f"{name}.pt"))
        pd.DataFrame(hist).to_csv(os.path.join(save_dir, f"history_{name}.csv"), index=False)
        pert = tensor.squeeze(0).cpu().numpy().transpose(1, 2, 0)
        pert_norm = (pert - pert.min()) / (pert.max() - pert.min() + 1e-12)
        Image.fromarray((pert_norm * 255).astype(np.uint8)).save(os.path.join(save_dir, f"{name}.png"))

    all_histories[model_name] = {'vanilla': hist_vanilla, 'ti': hist_ti, 'eot': hist_eot, 'both': hist_both}

    gc.collect()
    torch.cuda.empty_cache()

shutil.make_archive("/kaggle/working/ablation_uaps", 'zip', save_dir)

curves_dir = "/kaggle/working/loss_curves"
os.makedirs(curves_dir, exist_ok=True)

ablation_labels = {
    "vanilla": "Vanilla (No EOT, No TI)",
    "ti": "Only TI-FGSM",
    "eot": "Only EOT",
    "both": "Both TI & EOT"
}

for model_name, model_histories in all_histories.items():
    for ablation_key, title_suffix in ablation_labels.items():
        hist = model_histories[ablation_key]
        df = pd.DataFrame(hist)
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(df['epoch'], df['train_loss'], label='Train Loss', color='#1f77b4', linewidth=2.2, marker='o', markersize=4)
        ax.plot(df['epoch'], df['val_loss'], label='Val Loss', color='#ff7f0e', linewidth=2.2, marker='s', markersize=4)
        ax.set_title(f"Loss Progression - {model_name} - {title_suffix}", fontsize=13, fontweight='bold', pad=12)
        ax.set_xlabel("Epoch", fontsize=11, fontweight='bold')
        ax.set_ylabel("Loss (-Cosine Similarity)", fontsize=11, fontweight='bold')
        ax.grid(True, linestyle='--', alpha=0.6)
        ax.legend(fontsize=11, frameon=True)
        file_name = f"curve_{model_name}_{ablation_key}.png"
        plt.tight_layout()
        plt.savefig(os.path.join(curves_dir, file_name), dpi=300, bbox_inches='tight')
        plt.show()

shutil.make_archive("/kaggle/working/loss_curves", 'zip', curves_dir)
print(f"-> Đã lưu các đồ thị vào: /kaggle/working/loss_curves.zip")